In [1]:
pip install torch numpy pandas nltk


Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import re
import string
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import Dataset, DataLoader

CAPTIONS_FILE = "captions.txt"  
GLOVE_PATH    = "glove.6B.200d.txt"
EMBED_DIM     = 200
MIN_FREQ      = 2       
BATCH_SIZE    = 32

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", DEVICE)


def load_captions(captions_file):
    mapping = {}
    with open(captions_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if "\t" in line:                      
                img_part, caption = line.split("\t", 1)
                image_id = img_part.split("#")[0]
            else:                                    
                parts = line.split(",", 1)
                if len(parts) != 2:
                    continue
                image_id, caption = parts
                if image_id.lower() == "image":      
                    continue
            mapping.setdefault(image_id.strip(), []).append(caption.strip())
    return mapping


def clean_caption(caption):
    caption = caption.lower()
    caption = caption.translate(str.maketrans("", "", string.punctuation))
    caption = re.sub(r"\d+", "", caption)
    caption = re.sub(r"\s+", " ", caption).strip()
    return [w for w in caption.split() if len(w) > 1]

mapping = load_captions(CAPTIONS_FILE)
print(f"{len(mapping)} images chargées.")

# Aperçu
ex_id = next(iter(mapping))
print("Exemple :", ex_id)
for c in mapping[ex_id]:
    print("  brut   :", c)
    print("  nettoyé:", clean_caption(c))

Device : cpu
8091 images chargées.
Exemple : 1000268201_693b08cb0e.jpg
  brut   : A child in a pink dress is climbing up a set of stairs in an entry way .
  nettoyé: ['child', 'in', 'pink', 'dress', 'is', 'climbing', 'up', 'set', 'of', 'stairs', 'in', 'an', 'entry', 'way']
  brut   : A girl going into a wooden building .
  nettoyé: ['girl', 'going', 'into', 'wooden', 'building']
  brut   : A little girl climbing into a wooden playhouse .
  nettoyé: ['little', 'girl', 'climbing', 'into', 'wooden', 'playhouse']
  brut   : A little girl climbing the stairs to her playhouse .
  nettoyé: ['little', 'girl', 'climbing', 'the', 'stairs', 'to', 'her', 'playhouse']
  brut   : A little girl in a pink dress going into a wooden cabin .
  nettoyé: ['little', 'girl', 'in', 'pink', 'dress', 'going', 'into', 'wooden', 'cabin']


In [10]:
# Definition du vocabulaire avec : pad pour remplissage des séquences, unk pour les mots inconnus ou fautes et sos/eos pour le debut et la fin de la séquence

PAD, UNK, SOS, EOS = "<pad>", "<unk>", "<sos>", "<eos>"


class Vocabulary:
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.itos = {0: PAD, 1: UNK, 2: SOS, 3: EOS}
        self.stoi = {PAD: 0, UNK: 1, SOS: 2, EOS: 3}

    def build(self, list_of_token_lists):
        counter = Counter()
        for tokens in list_of_token_lists:
            counter.update(tokens)
        idx = len(self.itos)
        for word, freq in counter.items():
            if freq >= self.min_freq:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1

    def encode(self, tokens, add_special=True):
        ids = [self.stoi.get(t, self.stoi[UNK]) for t in tokens]
        if add_special:
            ids = [self.stoi[SOS]] + ids + [self.stoi[EOS]]
        return ids

    def __len__(self):
        return len(self.itos)

all_tokens = [clean_caption(c) for caps in mapping.values() for c in caps]

vocab = Vocabulary(min_freq=MIN_FREQ)
vocab.build(all_tokens)
print(f"Taille du vocabulaire : {len(vocab)}")

Taille du vocabulaire : 5193


In [12]:
# Construction de la matrice d'embedding à partir de Glove

def build_embedding_matrix(vocab, glove_path, embed_dim=200):
    matrix = np.random.normal(scale=0.6, size=(len(vocab), embed_dim)).astype(np.float32)
    matrix[vocab.stoi[PAD]] = 0.0

    found = 0
    if os.path.exists(glove_path):
        with open(glove_path, "r", encoding="utf-8") as f:
            for line in f:
                values = line.split()
                word = values[0]
                if word in vocab.stoi:
                    matrix[vocab.stoi[word]] = np.asarray(values[1:], dtype=np.float32)
                    found += 1
        print(f"GloVe : {found}/{len(vocab)} mots trouvés")
    else:
        print(f"[!] {glove_path} introuvable — embeddings aléatoires")
    return torch.tensor(matrix)


emb_matrix = build_embedding_matrix(vocab, GLOVE_PATH, EMBED_DIM)
print("Forme de la matrice d'embeddings :", emb_matrix.shape)

GloVe : 5039/5193 mots trouvés
Forme de la matrice d'embeddings : torch.Size([5193, 200])


In [13]:
# Transformation des captions en tenseurs pytorch
class CaptionDataset(Dataset):
    def __init__(self, mapping, vocab):
        self.vocab = vocab
        self.samples = []
        for image_id, captions in mapping.items():
            for cap in captions:
                tokens = clean_caption(cap)
                if tokens:
                    self.samples.append((image_id, self.vocab.encode(tokens)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_id, ids = self.samples[idx]
        return image_id, torch.tensor(ids, dtype=torch.long)


def collate_fn(batch):
    image_ids, sequences = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences])
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    return list(image_ids), padded, lengths


dataset = CaptionDataset(mapping, vocab)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
print(f"{len(dataset)} couples image-caption.")

40453 couples image-caption.


In [14]:
# LE RNN
class TextEncoder(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=256, num_layers=1,
                 rnn_type="lstm", bidirectional=True, dropout=0.3,
                 freeze_embeddings=False):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix, freeze=freeze_embeddings, padding_idx=0
        )

        rnn_cls = nn.LSTM if rnn_type.lower() == "lstm" else nn.GRU
        self.rnn_type = rnn_type.lower()
        self.rnn = rnn_cls(
            input_size=embed_dim, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.bidirectional = bidirectional
        self.output_dim = hidden_dim * (2 if bidirectional else 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, captions, lengths):
        embedded = self.dropout(self.embedding(captions))
        packed = pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        if self.rnn_type == "lstm":
            _, (hidden, _) = self.rnn(packed)
        else:
            _, hidden = self.rnn(packed)

        if self.bidirectional:
            sentence_repr = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            sentence_repr = hidden[-1]
        return self.dropout(sentence_repr)

encoder = TextEncoder(emb_matrix, hidden_dim=256, rnn_type="lstm",
                      bidirectional=True).to(DEVICE)
print(encoder)
print("\nDimension d'encodage texte :", encoder.output_dim)

TextEncoder(
  (embedding): Embedding(5193, 200, padding_idx=0)
  (rnn): LSTM(200, 256, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
)

Dimension d'encodage texte : 512


In [15]:
#TEST

image_ids, captions, lengths = next(iter(loader))
captions, lengths = captions.to(DEVICE), lengths.to(DEVICE)

with torch.no_grad():
    text_features = encoder(captions, lengths)

print("Captions (padded)  :", captions.shape)
print("Encodage texte     :", text_features.shape)  

Captions (padded)  : torch.Size([32, 18])
Encodage texte     : torch.Size([32, 512])
